## 5. Segmentation via TensorFlow U-Net
Next, we segment the pits using a pre-trained U-Net model. 

*Note: The model expects a custom loss function (`dice_focal_loss`) to be passed during loading, even if we only run inference. We provide a dummy function to satisfy the keras loader requirements.

In [2]:
import sys
import platform

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Architecture:", platform.machine())

Python executable: /Users/polinabobrova/Desktop/GaN Quantum Dot Analysis/venv-ml/bin/python
Python version: 3.10.20 (main, Mar  3 2026, 00:49:35) [Clang 17.0.0 (clang-1700.6.4.2)]
Architecture: arm64


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import skimage

print("numpy:", np.__version__)
print("tensorflow:", tf.__version__)
print("skimage:", skimage.__version__)
print("devices:", tf.config.list_physical_devices())
print("gpus:", tf.config.list_physical_devices("GPU"))

numpy: 1.26.4
tensorflow: 2.16.2
skimage: 0.25.2
devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
gpus: []


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import gwyfile

afm_data = np.load("data_numpy/1nm_QD_AFMs_AIX6468_along_curved_edge_1um_100pN_18gain.0_00000/channel_00___0_data.npy")

In [6]:
import copy
from scipy.ndimage import median_filter

# topo_corrected = afm_data.copy()
topo_corrected = np.copy(afm_data)

# 1. Row alignment (subtract median of each row)
topo_corrected -= np.median(topo_corrected, axis=1)[:, np.newaxis]

# 2. Scar removal using a median filter (simple approach)
topo_corrected = median_filter(topo_corrected, size=(1, 5))

# 3. 2D plane fit (flatten plane)
X, Y = np.meshgrid(np.arange(topo_corrected.shape[1]), np.arange(topo_corrected.shape[0]))
A = np.column_stack((X.ravel(), Y.ravel(), np.ones(X.size)))
C, _, _, _ = np.linalg.lstsq(A, topo_corrected.ravel(), rcond=None)
plane = (A @ C).reshape(topo_corrected.shape)
topo_corrected -= plane

# topo_corrected is now ready for segmentation
afm_data = topo_corrected

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import filters, measure
from tensorflow import keras

# Define custom loss function fallback required for model loading
def dice_focal_loss(y_true, y_pred): 
    return 0

# Model path
unet_path = r"segmentation_model.keras"

try:
    # Load model without compiling, injecting the custom loss object
    unet_model = keras.models.load_model(
        unet_path, 
        compile=False, 
        custom_objects={"dice_focal_loss": dice_focal_loss}
    )
    print("U-Net loaded successfully.")
    
    # Preprocess image: format shape to (1, H, W, 1) and cast to float32
    img_tensor = afm_data.astype(np.float32)[np.newaxis, ..., np.newaxis]
    
    # Run prediction
    pred_mask = unet_model.predict(img_tensor, verbose=0)
    
    # Threshold the sigmoid output to create a strict binary mask
    binary_mask_unet = (pred_mask[0, ..., 0] > 0.5).astype(np.uint8)
    
    # Visualize U-Net Results
    fig, (ax1, ax2) = plt.subplots(1, 2)
    ax1.imshow(afm_data, cmap='afmhot')
    ax1.set_title("Corrected AFM")
    ax1.axis('off')

    ax2.imshow(binary_mask_unet, cmap='gray')
    ax2.set_title("U-Net Segmentation")
    ax2.axis('off')
    plt.show()
    
except Exception as e:
    print(f"Failed to load or run U-Net. Ensure {unet_path} is in the current directory.")
    print(f"Error: {e}")

Failed to load or run U-Net. Ensure segmentation_model.keras is in the current directory.
Error: File not found: filepath=segmentation_model.keras. Please ensure the file is an accessible `.keras` zip file.


Making decision boundaries for the dots, dots within defects and any interesting artefacts. 

Making a separate cost function for all of these? 